In [1]:
import os
import multiprocessing
from collections import defaultdict
import datetime
import gc
import re
import sqlite3
import string
import time


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.spatial import ConvexHull
from threadpoolctl import threadpool_limits
from tqdm import tqdm

# ----------------------------------------------------------------------
# Parallel / BLAS configuration
# ----------------------------------------------------------------------
num_logical_cores = os.cpu_count()
if num_logical_cores:
    os.environ["OMP_NUM_THREADS"] = str(num_logical_cores)
else:
    os.environ["OMP_NUM_THREADS"] = "1"

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
GENES_TABLE = "attributes"
NEIGHBORS_TABLE = "neighbors"

COL_NEIGHBORHOOD_ID = "organism"
COL_GENE_ID = "id"
COL_LINKING_KEY = "id"
COL_ACCESSION_ID = "accession"
COL_FUNCTION_DESC = "desc"
COL_PFAM_IDS = "family"
COL_INTERPRO_IDS = "ipro_family"
COL_REL_START = "rel_start"
COL_REL_STOP = "rel_stop"
COL_SSN_CLUSTER_ID = "cluster_num"

HIT_GENE_WEIGHT_FACTOR = 10
DIRECT_NEIGHBOR_WEIGHT_FACTOR = 3
MAX_NEIGHBORS_PER_SIDE = 10

DEFAULT_SSN_CLUSTER_VALUE_TO_FILTER = [None, 0]

SAVE_PLOTS = True
OUTPUT_DIR = "gnn_cluster_plots_circular"
REPORT_FILENAME_BASE = "gnn_clustering_report"
OUTPUT_FORMATS = ["pdf"]
DPI = 600
HIGHLIGHT_COLOR = "red"

COLLAPSE_IDENTICAL_NEIGHBORHOODS = True
COLLAPSE_CORE_SIMILARITY_THRESHOLD = 0.0
COLLAPSE_FULL_NEIGHBORHOOD_SIMILARITY_THRESHOLD = 0.3

MIN_ITEMS_FOR_PARALLEL_PROCESSING = 20

# Precompiled regex / constants
_IPR_REGEX = re.compile(r"IPR\d+", re.IGNORECASE)
_PFAM_REGEX = re.compile(r"PF\d+", re.IGNORECASE)
_UNINFORMATIVE_TERMS = frozenset(["none", "", "null", "uncharacterized protein"])


# ----------------------------------------------------------------------
# Basic feature parsing
# ----------------------------------------------------------------------
def parse_annotation_string(annotation_str, prefix=""):
    if not isinstance(annotation_str, str) or pd.isna(annotation_str):
        return set()
    if annotation_str.lower().strip() in _UNINFORMATIVE_TERMS:
        return set()

    features = set()
    parts = [p.strip() for p in re.split(r"[-;]", annotation_str) if p.strip()]

    for part in parts:
        if part.lower().strip() in _UNINFORMATIVE_TERMS:
            continue

        if _IPR_REGEX.match(part):
            features.add(f"{prefix}{part.upper()}")
        elif _PFAM_REGEX.match(part):
            features.add(f"{prefix}{part.upper()}")
        else:
            clean_part = re.sub(r"\s+", " ", part).lower().strip()
            if clean_part:
                features.add(f"{prefix}{clean_part}")
    return features


def extract_features_from_gene_row(
    gene_row,
    current_weight_factor=1,
    base_prefix="N_",
    include_desc=True,
    include_pfam=True,
    include_interpro=True,
):
    raw_features = set()

    if include_desc:
        raw_features.update(parse_annotation_string(gene_row[COL_FUNCTION_DESC]))
    if include_pfam:
        raw_features.update(parse_annotation_string(gene_row[COL_PFAM_IDS]))
    if include_interpro:
        raw_features.update(parse_annotation_string(gene_row[COL_INTERPRO_IDS]))

    if current_weight_factor <= 1:
        return {f"{base_prefix}{f}" for f in raw_features}

    features = set()
    for f in raw_features:
        for i in range(current_weight_factor):
            features.add(f"{base_prefix}{f}_w{i}")
    return features


def _coerce_numeric(value):
    if value is None or pd.isna(value):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _limit_neighbors_per_side(
    neighbor_rows,
    col_rel_start=COL_REL_START,
    col_rel_stop=COL_REL_STOP,
    max_neighbors_per_side=MAX_NEIGHBORS_PER_SIDE,
):
    if max_neighbors_per_side is None or max_neighbors_per_side <= 0:
        return list(neighbor_rows)

    left_side = []
    right_side = []
    unknown_side = []

    for idx, nrow in enumerate(neighbor_rows):
        rel_start = _coerce_numeric(nrow.get(col_rel_start))
        rel_stop = _coerce_numeric(nrow.get(col_rel_stop))

        if rel_stop is not None and rel_stop < 0:
            # For left-side neighbors, values closer to 0 are closer to the hit gene.
            left_side.append((rel_stop, idx, nrow))
        elif rel_start is not None and rel_start > 0:
            # For right-side neighbors, smaller positive values are closer to the hit gene.
            right_side.append((rel_start, idx, nrow))
        else:
            unknown_side.append((idx, nrow))

    left_keep = sorted(left_side, key=lambda x: x[0], reverse=True)[:max_neighbors_per_side]
    right_keep = sorted(right_side, key=lambda x: x[0])[:max_neighbors_per_side]

    selected = [(idx, row) for _, idx, row in left_keep]
    selected.extend((idx, row) for _, idx, row in right_keep)
    selected.extend(unknown_side)

    # Keep stable input order for deterministic downstream behavior.
    selected.sort(key=lambda x: x[0])
    return [row for _, row in selected]


# ----------------------------------------------------------------------
# Distance calculation (Jaccard) with optional parallelism
# ----------------------------------------------------------------------
def parallel_pdist_jaccard(feature_matrix, num_cores=-1):
    """
    Compute condensed Jaccard distance for a binary sparse matrix (CSR).
    """
    if not isinstance(feature_matrix, sp.csr_matrix):
        if isinstance(feature_matrix, (sp.csc_matrix, sp.lil_matrix, sp.coo_matrix)):
            feature_matrix = feature_matrix.tocsr()
        else:
            raise TypeError("feature_matrix must be a SciPy sparse matrix")

    n_samples = feature_matrix.shape[0]
    if n_samples <= 1:
        return np.array([])

    if num_cores == -1:
        detected = os.cpu_count()
        num_cores = detected if detected and detected > 0 else 1
    elif num_cores == 0:
        num_cores = 1

    # Precompute row index sets once
    print(f"  Pre-computing feature sets for N={n_samples} ...")
    t0 = time.time()
    feature_sets = [
        set(feature_matrix.indices[feature_matrix.indptr[i] : feature_matrix.indptr[i + 1]])
        for i in tqdm(range(n_samples), desc="  Feature sets", leave=False)
    ]
    print(f"  Done in {time.time() - t0:.2f}s")
    del feature_matrix
    gc.collect()

    def _dist_chunk(start_i, end_i, sets_ref, n_total):
        with threadpool_limits(limits=1, user_api="blas"):
            chunk = []
            for i in range(start_i, end_i):
                si = sets_ref[i]
                for j in range(i + 1, n_total):
                    sj = sets_ref[j]
                    inter = len(si & sj)
                    union = len(si | sj)
                    d = 0.0 if union == 0 else 1.0 - inter / union
                    chunk.append(d)
            return chunk

    total_i = n_samples - 1
    if total_i <= 0:
        return np.array([])

    if n_samples < MIN_ITEMS_FOR_PARALLEL_PROCESSING or num_cores == 1:
        print(f"  Using sequential Jaccard (N={n_samples})")
        res = [_dist_chunk(0, total_i, feature_sets, n_samples)]
    else:
        print(f"  Using parallel Jaccard (N={n_samples}, cores={num_cores})")
        num_tasks = min(total_i, num_cores * 4)
        chunk_size = max(1, (total_i + num_tasks - 1) // num_tasks)
        ranges = [(k, min(k + chunk_size, total_i)) for k in range(0, total_i, chunk_size)]

        tasks = [
            delayed(_dist_chunk)(start, end, feature_sets, n_samples) for start, end in ranges
        ]
        res = Parallel(n_jobs=num_cores, backend="loky", verbose=0)(tasks)

    return np.concatenate(res)


def _compute_silhouette_like_scores_from_sparse_binary(feature_matrix, cluster_assignments, max_n=300):
    """
    Compute a Jaccard-based silhouette-like score per sample.
    Returns NaN scores when not enough clusters exist or when group size is too large.
    """
    n = feature_matrix.shape[0]
    scores = np.full(n, np.nan, dtype=float)
    if n < 2 or n > max_n:
        return scores

    labels = np.asarray(cluster_assignments)
    unique_labels = np.unique(labels)
    if unique_labels.size < 2:
        return scores

    sets = [
        set(feature_matrix.indices[feature_matrix.indptr[i] : feature_matrix.indptr[i + 1]])
        for i in range(n)
    ]

    dist_mat = np.zeros((n, n), dtype=float)
    for i in range(n):
        si = sets[i]
        for j in range(i + 1, n):
            sj = sets[j]
            inter = len(si & sj)
            union = len(si | sj)
            d = 0.0 if union == 0 else 1.0 - (inter / union)
            dist_mat[i, j] = d
            dist_mat[j, i] = d

    idx_by_label = {lab: np.where(labels == lab)[0] for lab in unique_labels}
    for i in range(n):
        li = labels[i]
        same = idx_by_label[li]
        same = same[same != i]
        a = float(np.mean(dist_mat[i, same])) if same.size > 0 else 0.0

        b = np.inf
        for lj, members in idx_by_label.items():
            if lj == li or members.size == 0:
                continue
            b = min(b, float(np.mean(dist_mat[i, members])))

        if not np.isfinite(b):
            scores[i] = np.nan
        else:
            denom = max(a, b)
            scores[i] = 0.0 if denom == 0 else (b - a) / denom

    return scores


# ----------------------------------------------------------------------
# Collapsing similar neighborhoods
# ----------------------------------------------------------------------
def _perform_collapsing(
    all_neighborhood_features,
    full_neighborhood_labels_map,
    core_neighborhood_features,
    collapse_core_similarity_threshold,
    collapse_full_neighborhood_similarity_threshold,
    output_prefix="",
    report_file=None,
    parallelize_pdist=False,
):

    def log(msg):
        print(msg)
        if report_file:
            report_file.write(msg + "\n")

    log(
        f"{output_prefix}  Collapsing neighborhoods (core thr: {collapse_core_similarity_threshold}, "
        f"full thr: {collapse_full_neighborhood_similarity_threshold})"
    )
    t0_all = time.time()
    collapsed_groups_report = {}

    # ---- Stage 1: core-based grouping ----
    labels = sorted(core_neighborhood_features.keys())
    if len(labels) < 2:
        log(f"{output_prefix}  <2 neighborhoods; skipping collapsing.")
        return all_neighborhood_features, full_neighborhood_labels_map, collapsed_groups_report

    core_vocab = sorted(set.union(*core_neighborhood_features.values()))
    if not core_vocab:
        log(f"{output_prefix}  No core features; skipping collapsing.")
        return all_neighborhood_features, full_neighborhood_labels_map, collapsed_groups_report

    log(f"{output_prefix}  Stage1: building core sparse matrix")
    feat_to_idx = {f: i for i, f in enumerate(core_vocab)}
    n_nh = len(labels)
    n_feat = len(core_vocab)
    mat_lil = sp.lil_matrix((n_nh, n_feat), dtype=np.int8)
    for i, nh in enumerate(tqdm(labels, desc=f"{output_prefix}  Core features", leave=False)):
        for f in core_neighborhood_features[nh]:
            j = feat_to_idx[f]
            mat_lil[i, j] = 1
    core_mat = mat_lil.tocsr()
    del mat_lil
    gc.collect()

    if core_mat.shape[0] < 2:
        log(f"{output_prefix}  <2 core vectors; skipping collapsing.")
        return all_neighborhood_features, full_neighborhood_labels_map, collapsed_groups_report

    # Check if all identical
    if n_nh > 1 and all((core_mat[0] != core_mat[i]).nnz == 0 for i in range(1, n_nh)):
        # everything identical -> one group
        pre_clusters = {labels[i]: 1 for i in range(len(labels))}
        log(f"{output_prefix}  All core vectors identical; treating as one initial group.")
    else:
        log(f"{output_prefix}  Stage1: computing core Jaccard distances")
        t0 = time.time()
        dists = parallel_pdist_jaccard(core_mat, num_cores=-1 if parallelize_pdist else 1)
        log(f"{output_prefix}  Stage1: distance calc in {time.time() - t0:.2f}s")
        from scipy.cluster.hierarchy import linkage, fcluster

        link_core = linkage(dists, method="average")
        pre_clusters_array = fcluster(link_core, collapse_core_similarity_threshold, criterion="distance")
        pre_clusters = {labels[i]: pre_clusters_array[i] for i in range(len(labels))}

    initial_core_groups = defaultdict(list)
    for nh, cid in pre_clusters.items():
        initial_core_groups[cid].append(nh)

    log(
        f"{output_prefix}  Stage1: {len(initial_core_groups)} core groups "
        f"({time.time() - t0_all:.2f}s partial)"
    )

    del core_mat
    if "dists" in locals():
        del dists
    gc.collect()

    # ---- Stage 2: full neighborhood collapsing within each core group ----
    def gen_letter(idx):
        if idx < 26:
            return string.ascii_uppercase[idx]
        first = (idx // 26) - 1
        second = idx % 26
        return f"{string.ascii_uppercase[first]}{string.ascii_uppercase[second]}"

    log(f"{output_prefix}  Stage2: full-neighborhood collapsing")

    def process_core_group_chunk(
        group_ids,
        all_feat_ref,
        labels_map_ref,
        thr_full,
        allow_parallel,
    ):
        out = []
        collapsed_count = 0
        cores_inner = -1 if allow_parallel else 1

        with threadpool_limits(limits=1, user_api="blas"):
            for gid in group_ids:
                members = initial_core_groups[gid]
                if len(members) < 2:
                    m = members[0]
                    out.append((m, all_feat_ref[m], labels_map_ref[m], None))
                    continue

                vocab = sorted(set.union(*[all_feat_ref[m] for m in members]))
                if not vocab:
                    assignments = {m: 1 for m in members}
                else:
                    ft_idx = {f: i for i, f in enumerate(vocab)}
                    n_m = len(members)
                    m_lil = sp.lil_matrix((n_m, len(vocab)), dtype=np.int8)
                    for i, nh in enumerate(members):
                        for f in all_feat_ref[nh]:
                            j = ft_idx[f]
                            m_lil[i, j] = 1
                    mat = m_lil.tocsr()

                    if n_m > 1 and all((mat[0] != mat[i]).nnz == 0 for i in range(1, n_m)):
                        assignments = {m: 1 for m in members}
                    else:
                        d = parallel_pdist_jaccard(mat, num_cores=cores_inner)
                        if d.size == 0:
                            assignments = {m: 1 for m in members}
                        else:
                            link = linkage(d, method="average")
                            arr = fcluster(link, thr_full, criterion="distance")
                            assignments = {members[i]: arr[i] for i in range(len(members))}
                    del mat
                    if "d" in locals():
                        del d
                    if "link" in locals():
                        del link
                    gc.collect()

                groups = defaultdict(list)
                for nh, cid in assignments.items():
                    groups[cid].append(nh)

                for cid, collapsed_members in sorted(groups.items()):
                    if len(collapsed_members) > 1:
                        collapsed_count += len(collapsed_members) - 1
                        rep = collapsed_members[0]
                        union_features = set()
                        for nh in collapsed_members:
                            union_features.update(all_feat_ref[nh])
                        out.append((rep, union_features, labels_map_ref[rep], collapsed_members))
                    else:
                        m = collapsed_members[0]
                        out.append((m, all_feat_ref[m], labels_map_ref[m], None))

        return out, collapsed_count

    all_ids = sorted(initial_core_groups.keys())
    n_core_groups = len(all_ids)
    if n_core_groups < MIN_ITEMS_FOR_PARALLEL_PROCESSING or not parallelize_pdist:
        chunk_results = [
            process_core_group_chunk(
                all_ids,
                all_neighborhood_features,
                full_neighborhood_labels_map,
                collapse_full_neighborhood_similarity_threshold,
                False,
            )
        ]
    else:
        n_cores = os.cpu_count() or 1
        chunk_size = max(1, n_core_groups // n_cores)
        chunks = [all_ids[i : i + chunk_size] for i in range(0, n_core_groups, chunk_size)]
        chunk_results = Parallel(n_jobs=n_cores, backend="loky", verbose=0)(
            delayed(process_core_group_chunk)(
                ch,
                all_neighborhood_features,
                full_neighborhood_labels_map,
                collapse_full_neighborhood_similarity_threshold,
                parallelize_pdist,
            )
            for ch in chunks
        )

    final_neighborhood_features = {}
    final_neighborhood_labels_map = {}
    collapsed_total = 0
    letter_counter = 0

    for res, local_count in chunk_results:
        collapsed_total += local_count
        for rep_label, feats, label_entry, collapsed_members in res:
            if collapsed_members is not None:
                code = gen_letter(letter_counter)
                letter_counter += 1
                org, hit_id, ssn_id, acc, _ = label_entry
                final_neighborhood_labels_map[rep_label] = (
                    org,
                    hit_id,
                    ssn_id,
                    acc,
                    (len(collapsed_members), code),
                )
                collapsed_groups_report[code] = {
                    "representative": rep_label,
                    "members": sorted(collapsed_members),
                    "count": len(collapsed_members),
                }
                final_neighborhood_features[rep_label] = feats
            else:
                final_neighborhood_features[rep_label] = feats
                final_neighborhood_labels_map[rep_label] = label_entry

    if collapsed_total > 0:
        log(
            f"{output_prefix}  Collapsed {collapsed_total} neighborhoods -> "
            f"{len(final_neighborhood_features)} unique entries "
            f"({time.time() - t0_all:.2f}s total)."
        )
    else:
        log(f"{output_prefix}  No collapsing performed ({time.time() - t0_all:.2f}s).")

    return final_neighborhood_features, final_neighborhood_labels_map, collapsed_groups_report


# ----------------------------------------------------------------------
# Circular phylogram plotting
# ----------------------------------------------------------------------
def _hierarchy_to_children(linkage_matrix, labels):
    """
    Build adjacency (children) list from scipy linkage matrix.
    Leaf node ids: 0..(n-1)
    Internal node ids: n..(n+linkage_matrix.shape[0]-1)
    """
    n_leaves = len(labels)
    children = {}
    for i, (c1, c2, dist, _) in enumerate(linkage_matrix):
        node_id = n_leaves + i
        children[node_id] = (int(c1), int(c2))
    node_ids = set(range(n_leaves)) | set(children.keys())
    return children, node_ids


def _gather_descendant_leaves(node_id, children, n_leaves, cache=None):
    """
    Return list of leaf node indices descending from node_id.
    """
    if cache is None:
        cache = {}
    if node_id in cache:
        return cache[node_id]

    if node_id < n_leaves:
        cache[node_id] = [node_id]
        return cache[node_id]

    c1, c2 = children[node_id]
    leaves = _gather_descendant_leaves(c1, children, n_leaves, cache) + \
             _gather_descendant_leaves(c2, children, n_leaves, cache)
    cache[node_id] = leaves
    return leaves


def _collect_cluster_colors(cluster_assignments, labels, cmap_name="tab20"):
    clusters = sorted(set(cluster_assignments))
    cmap = plt.get_cmap(cmap_name, len(clusters))
    cluster_to_color = {cid: cmap(i) for i, cid in enumerate(clusters)}
    label_to_color = {}
    for label_idx, cid in enumerate(cluster_assignments):
        label_to_color[labels[label_idx]] = cluster_to_color[cid]
    return cluster_to_color, label_to_color


def _compute_dual_scale_radii(
    feature_matrix, 
    linkage_matrix, 
    labels, 
    inner_boundary=0.6, 
    consensus_threshold=0.25,
    leaf_stretch_power=1.5
):
    n_leaves = len(labels)
    n_internal = linkage_matrix.shape[0]

    # 1. Consensus Ancestor Logic
    col_counts = np.array((feature_matrix > 0).sum(axis=0)).flatten()
    threshold = n_leaves * consensus_threshold
    mca_vec = (col_counts >= threshold).astype(np.int8)

    leaf_div = np.zeros(n_leaves)
    for i in range(n_leaves):
        leaf_vec = feature_matrix[i].toarray().flatten().astype(np.int8)
        intersection = np.sum(np.logical_and(leaf_vec, mca_vec))
        union = np.sum(np.logical_or(leaf_vec, mca_vec))
        leaf_div[i] = 1.0 - (intersection / union) if union != 0 else 1.0

    max_div = np.max(leaf_div) if np.max(leaf_div) > 0 else 1.0
    norm_leaf_div = leaf_div / max_div

    # 2. Tree Scale (Inner branches remain linear relative to height)
    max_h = linkage_matrix[-1, 2] if n_internal > 0 else 1.0
    final_radii = {}
    for i in range(n_internal):
        node_id = n_leaves + i
        h = linkage_matrix[i, 2]
        progress = 1.0 - (h / max_h) 
        final_radii[node_id] = progress * inner_boundary

    # 3. Leaf Scale (Non-linear Stretch)
    children, _ = _hierarchy_to_children(linkage_matrix, labels)
    parent_map = {c1: p_id for p_id, (c1, c2) in children.items()}
    parent_map.update({c2: p_id for p_id, (c1, c2) in children.items()})

    for i in range(n_leaves):
        parent_id = parent_map.get(i)
        parent_r = final_radii[parent_id] if parent_id is not None else 0
        
        # Apply the Non-Linear Power Scale here
        # This only affects the distance from inner_boundary to 1.0
        stretched_div = norm_leaf_div[i] ** leaf_stretch_power
        
        proposed_r = inner_boundary + (stretched_div * (1.0 - inner_boundary))
        final_radii[i] = max(proposed_r, parent_r + 0.01)

    return final_radii


def _compute_tree_metrics(linkage_matrix, labels):
    """
    Computes root-to-node distances and subtree weights.
    Returns:
      node_root_dist: dict of {node_id: distance_from_root}
      subtree_weight: dict of {node_id: sum_of_internal_distances} (used for angular spacing)
    """
    n_leaves = len(labels)
    n_total = n_leaves + linkage_matrix.shape[0]
    children, node_ids = _hierarchy_to_children(linkage_matrix, labels)
    
    # Height of nodes from linkage (distance at which clusters merge)
    # heights[leaf] = 0; heights[root] = max_dist
    heights = {i: 0.0 for i in range(n_leaves)}
    for i, (_, _, dist, _) in enumerate(linkage_matrix):
        heights[n_leaves + i] = float(dist)

    root_id = n_total - 1
    node_root_dist = {root_id: 0.0}
    subtree_weight = {i: 1.0 for i in range(n_leaves)} # Base weight for leaves

    # Breadth-first or Depth-first to get root-to-node distances
    # Distance = parent_height - child_height is the standard branch length
    stack = [root_id]
    while stack:
        curr = stack.pop()
        if curr in children:
            c1, c2 = children[curr]
            # Branch length is the difference in linkage 'age'
            branch_l1 = heights[curr] - heights[c1]
            branch_l2 = heights[curr] - heights[c2]
            
            node_root_dist[c1] = node_root_dist[curr] + branch_l1
            node_root_dist[c2] = node_root_dist[curr] + branch_l2
            
            stack.extend([c1, c2])

    # Compute subtree weights (sum of distances) to drive angular distribution
    # We do this bottom-up
    for i in range(linkage_matrix.shape[0]):
        node_id = n_leaves + i
        c1, c2 = children[node_id]
        # Weight is proportional to the distance of the split + children weights
        subtree_weight[node_id] = subtree_weight[c1] + subtree_weight[c2] + linkage_matrix[i, 2]

    return node_root_dist, subtree_weight, heights


def _compute_cluster_aware_angles(linkage_matrix, labels, subtree_weight, gap_factor=0.05):
    """
    Assigns angles with explicit gaps between branches to separate clusters.
    gap_factor: percentage of the available arc to leave empty at each split.
    """
    n_leaves = len(labels)
    children, _ = _hierarchy_to_children(linkage_matrix, labels)
    root_id = n_leaves + linkage_matrix.shape[0] - 1
    angles = {}

    def assign_angle(node_id, theta_start, theta_end):
        if node_id < n_leaves:
            angles[node_id] = (theta_start + theta_end) / 2.0
            return

        c1, c2 = children[node_id]
        w1, w2 = subtree_weight[c1], subtree_weight[c2]
        
        # Calculate available arc after removing a gap
        total_arc = theta_end - theta_start
        current_gap = total_arc * gap_factor
        usable_arc = total_arc - current_gap
        
        # Split usable arc proportional to weights
        mid_point = theta_start + usable_arc * (w1 / (w1 + w2))
        
        # Branch 1 gets first part, Branch 2 starts after the gap
        assign_angle(c1, theta_start, mid_point)
        assign_angle(c2, mid_point + current_gap, theta_end)

    assign_angle(root_id, 0.0, 2 * np.pi * 0.95) # Leave a final gap at the end
    return angles


def _select_label_indices(
    leaf_labels,
    cluster_assignments,
    coords,
    labels_map=None,
    original_input_sequence_id=None,
    max_labels=28,
    min_labels_per_cluster=1,
    extra_labels_per_cluster=1,
    min_label_separation_deg=40.0,
):
    n = len(leaf_labels)
    if n == 0:
        return set()

    max_labels = max(1, min(int(max_labels), n))
    idx_to_radius = {i: coords[i][0] for i in range(n)}
    idx_to_theta = {i: coords[i][1] for i in range(n)}
    min_sep = np.radians(max(0.0, float(min_label_separation_deg)))

    by_cluster = defaultdict(list)
    for i, cid in enumerate(cluster_assignments):
        by_cluster[cid].append(i)

    selected = set()
    mandatory = set()

    def _ang_dist(a, b):
        d = abs(a - b) % (2 * np.pi)
        return min(d, 2 * np.pi - d)

    def _is_far_enough(candidate_idx):
        if min_sep <= 0 or not selected:
            return True
        c_th = idx_to_theta[candidate_idx]
        for j in selected:
            if _ang_dist(c_th, idx_to_theta[j]) < min_sep:
                return False
        return True

    # Always keep highlighted accession labeled when present.
    if labels_map and original_input_sequence_id:
        for i, key in enumerate(leaf_labels):
            entry = labels_map.get(key)
            if entry and len(entry) > 3 and entry[3] == original_input_sequence_id:
                mandatory.add(i)
                break

    # Ensure at least one representative per cluster (outermost leaf).
    for cid, idxs in by_cluster.items():
        if not idxs:
            continue
        ordered = sorted(idxs, key=lambda i: idx_to_radius[i], reverse=True)
        for i in ordered[:max(1, min_labels_per_cluster)]:
            mandatory.add(i)

    # Keep all mandatory labels even when they violate angular spacing.
    selected.update(mandatory)
    max_labels = max(max_labels, len(selected))

    # Add extra labels per cluster only when they are angularly separated.
    if extra_labels_per_cluster > 0:
        for cid, idxs in by_cluster.items():
            if len(selected) >= max_labels:
                break
            ranked = sorted(idxs, key=lambda i: (idx_to_radius[i], idx_to_theta[i]), reverse=True)
            added = 0
            for i in ranked:
                if i in selected:
                    continue
                if not _is_far_enough(i):
                    continue
                selected.add(i)
                added += 1
                if added >= extra_labels_per_cluster or len(selected) >= max_labels:
                    break

    # Fill remaining budget with globally outermost leaves under angular spacing.
    if len(selected) < max_labels:
        global_rank = sorted(range(n), key=lambda i: (idx_to_radius[i], -abs(np.pi - idx_to_theta[i])), reverse=True)
        for i in global_rank:
            if len(selected) >= max_labels:
                break
            if i in selected or not _is_far_enough(i):
                continue
            selected.add(i)

    return selected


def _draw_circular_tree_with_clusters(
    linkage_matrix,
    leaf_labels,
    cluster_assignments,
    label_to_cluster_color,
    original_input_sequence_id=None,
    labels_map=None,
    title=None,
    out_prefix=None,
    output_dir=".",
    output_formats=("pdf",),
    dpi=600,
    show=False,
    save_plots=True,
    feature_matrix=None,
    inner_boundary=0.5,    # Where tree ends
    consensus_threshold=0.25,     # If Features are in 25% of NHs they are "Ancestral"
    leaf_stretch_power=1.5, # Exaggerate leaf distances non-linearly
    gap_factor=0.08,
    max_labels=28,
    min_labels_per_cluster=1,
    extra_labels_per_cluster=1,
    min_label_separation_deg=40.0,
    min_label_angle_deg=6.0,
    max_stagger_levels=4,
    label_base_offset=0.035,
    label_stagger_step=0.02,
    draw_label_leaders=True,
    leader_min_offset=0.05,
    label_angle_jitter_deg=1.6,
):
    n_leaves = len(leaf_labels)
    
    # 1. Radii & Angles
    radii = _compute_dual_scale_radii(
        feature_matrix, linkage_matrix, leaf_labels, 
        inner_boundary=inner_boundary, consensus_threshold=consensus_threshold, leaf_stretch_power=leaf_stretch_power
    )
    _, subtree_weight, _ = _compute_tree_metrics(linkage_matrix, leaf_labels)
    angles = _compute_cluster_aware_angles(linkage_matrix, leaf_labels, subtree_weight, gap_factor)
    children, _ = _hierarchy_to_children(linkage_matrix, leaf_labels)
    
    # Coordinate mapping (Angles for internal nodes)
    coords = {}
    def get_coords(node_id):
        if node_id in coords: return coords[node_id]
        r = radii[node_id]
        if node_id < n_leaves:
            theta = angles[node_id]
        else:
            c1, c2 = children[node_id]
            _, th1 = get_coords(c1); _, th2 = get_coords(c2)
            theta = (th1 * subtree_weight[c1] + th2 * subtree_weight[c2]) / (subtree_weight[c1] + subtree_weight[c2])
        coords[node_id] = (r, theta)
        return coords[node_id]

    for nid in range(n_leaves + linkage_matrix.shape[0]): get_coords(nid)

    # Publication-quality figure setup
    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(20, 20))
    ax.set_theta_direction(-1)
    ax.set_theta_offset(np.pi / 2.0)
    ax.set_axis_off()
    
    # Set publication-ready style parameters
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts for publication
    plt.rcParams['ps.fonttype'] = 42

    # Subtle guide rings to improve depth perception without clutter.
    theta_full = np.linspace(0, 2 * np.pi, 300)
    ax.plot(theta_full, np.full_like(theta_full, inner_boundary), color=(0.75, 0.75, 0.75, 0.55), lw=0.8, ls='--', zorder=0)
    ax.plot(theta_full, np.full_like(theta_full, 1.0), color=(0.8, 0.8, 0.8, 0.35), lw=0.7, ls=':', zorder=0)

    # 2. Draw Branches with publication-quality thickness
    for node_id, (c1, c2) in children.items():
        r_p, th_p = coords[node_id]
        for child in (c1, c2):
            r_c, th_c = coords[child]
            
            # Default color with better contrast
            color = label_to_cluster_color[leaf_labels[child]] if child < n_leaves else (0.3, 0.3, 0.3, 0.6)
            linewidth = 2.3  # Thicker for publication
            zorder = 2

            # HIGHLIGHT LOGIC: If this branch leads directly to our target
            if child < n_leaves and labels_map:
                # labels_map[leaf_label] = (org, hit_id, ssn, accession, collapsed)
                acc = labels_map[leaf_labels[child]][3]
                if original_input_sequence_id and acc == original_input_sequence_id:
                    color = "#DC143C"  # Crimson for better publication contrast
                    linewidth = 3.5  # Much thicker for highlight
                    zorder = 10
            
            # Radial branch with smooth rendering
            ax.plot([th_c, th_c], [r_p, r_c], color=color, lw=linewidth, 
                    zorder=zorder, solid_capstyle='round', solid_joinstyle='round')
            
            # Connecting arc
            t_vals = np.linspace(min(th_p, th_c), max(th_p, th_c), 50)
            arc_color = color if child < n_leaves and zorder == 10 else (0.3, 0.3, 0.3, 0.6)
            ax.plot(t_vals, np.full_like(t_vals, r_p), color=arc_color, 
                    lw=linewidth * 0.75, zorder=zorder-1, solid_capstyle='round')

    # 3. Draw leaves and selectively annotate labels.
    selected_label_indices = _select_label_indices(
        leaf_labels,
        cluster_assignments,
        coords,
        labels_map=labels_map,
        original_input_sequence_id=original_input_sequence_id,
        max_labels=max_labels,
        min_labels_per_cluster=min_labels_per_cluster,
        extra_labels_per_cluster=extra_labels_per_cluster,
        min_label_separation_deg=min_label_separation_deg,
    )

    # Build collision-aware stagger levels from angular density.
    label_levels = {}
    sorted_selected = sorted(selected_label_indices, key=lambda i: coords[i][1])
    min_gap = np.radians(max(0.5, float(min_label_angle_deg)))
    prev_th = None
    level = 0
    for i in sorted_selected:
        th = coords[i][1]
        if prev_th is None:
            level = 0
        else:
            diff = th - prev_th
            if diff < min_gap:
                level += 1
            else:
                level = 0
        label_levels[i] = min(level, max(0, int(max_stagger_levels)))
        prev_th = th

    # Wrap-around correction between first and last angles.
    if len(sorted_selected) > 1:
        first_idx = sorted_selected[0]
        last_idx = sorted_selected[-1]
        wrap_diff = (coords[first_idx][1] + 2 * np.pi) - coords[last_idx][1]
        if wrap_diff < min_gap:
            label_levels[first_idx] = min(label_levels[first_idx] + 1, max(0, int(max_stagger_levels)))

    base_offset = max(0.0, float(label_base_offset))
    level_spacing = max(0.0, float(label_stagger_step))
    selected_thetas = {idx: coords[idx][1] for idx in selected_label_indices}
    max_leaf_r = max(radii[i] for i in range(n_leaves)) if n_leaves else 1.0
    plot_r_max = max_leaf_r + max(0.02, base_offset + max(0, int(max_stagger_levels)) * level_spacing)
    min_clearance_target = 0.030
    boundary_padding_px = 30.0
    x_axis_risk_window_deg = 58.0

    def _to_xy(theta_val, radius_val):
        return np.array([radius_val * np.cos(theta_val), radius_val * np.sin(theta_val)], dtype=float)

    def _max_inward_depth_for_text(text_len):
        # Allow deeper inward pull for long labels when boundary overflow persists.
        return 0.065 + min(0.115, 0.0030 * float(text_len))

    def _min_self_leaf_clearance(text_len):
        # Keep label anchors from collapsing onto their own leaf marker.
        return 0.026 + min(0.014, 0.00022 * float(text_len))

    # Occupancy map from nodes and branch polylines for local free-space checks.
    occupied_xy_list = []
    for li in range(n_leaves):
        lr, lth = coords[li]
        occupied_xy_list.append(_to_xy(lth, lr))
    for node_id, (c1, c2) in children.items():
        r_p, th_p = coords[node_id]
        for child in (c1, c2):
            r_c, th_c = coords[child]
            for rr in np.linspace(min(r_p, r_c), max(r_p, r_c), 6):
                occupied_xy_list.append(_to_xy(th_c, rr))
            t_occ = np.linspace(min(th_p, th_c), max(th_p, th_c), 8)
            for tt in t_occ:
                occupied_xy_list.append(_to_xy(tt, r_p))
    occupied_xy = np.vstack(occupied_xy_list) if occupied_xy_list else np.empty((0, 2), dtype=float)
    placed_label_xy = []

    def _clearance_at(theta_val, radius_val):
        c_xy = _to_xy(theta_val, radius_val)
        if occupied_xy.shape[0] > 0:
            geom_clear = float(np.min(np.linalg.norm(occupied_xy - c_xy, axis=1)))
        else:
            geom_clear = 1.0
        if placed_label_xy:
            arr = np.asarray(placed_label_xy)
            label_clear = float(np.min(np.linalg.norm(arr - c_xy, axis=1)))
        else:
            label_clear = 1.0
        return min(geom_clear, label_clear)

    # Renderer-aware geometry to check real text boxes against the circular frame.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    center_px = np.asarray(ax.transData.transform((0.0, 0.0)), dtype=float)
    rim_px = np.asarray(ax.transData.transform((0.0, plot_r_max)), dtype=float)
    boundary_radius_px = float(np.linalg.norm(rim_px - center_px))

    def _circle_overflow_px(bbox):
        corners = np.array(
            [[bbox.x0, bbox.y0], [bbox.x0, bbox.y1], [bbox.x1, bbox.y0], [bbox.x1, bbox.y1]],
            dtype=float,
        )
        dists = np.linalg.norm(corners - center_px, axis=1)
        effective_boundary_radius_px = max(1.0, boundary_radius_px - float(boundary_padding_px))
        return max(0.0, float(np.max(dists) - effective_boundary_radius_px))

    def _point_to_bbox_distance(px, py, bbox):
        dx = max(bbox.x0 - px, 0.0, px - bbox.x1)
        dy = max(bbox.y0 - py, 0.0, py - bbox.y1)
        return float(np.hypot(dx, dy))

    def _probe_bbox_metrics(theta_val, radius_val, text, ha_mode, va_mode, fontsize_val, pad_val, leaf_theta, leaf_radius):
        probe = ax.text(
            theta_val,
            radius_val,
            text,
            fontsize=fontsize_val,
            rotation=0.0,
            rotation_mode='anchor',
            ha=ha_mode,
            va=va_mode,
            bbox=dict(boxstyle=f'round,pad={pad_val:.2f}', facecolor='none', edgecolor='none', alpha=0.0),
            visible=False,
            zorder=-10,
        )
        bbox = probe.get_window_extent(renderer=renderer)
        probe.remove()
        leaf_px = np.asarray(ax.transData.transform((leaf_theta, leaf_radius)), dtype=float)
        overflow_px = _circle_overflow_px(bbox)
        leaf_box_dist_px = _point_to_bbox_distance(leaf_px[0], leaf_px[1], bbox)
        return overflow_px, leaf_box_dist_px

    for i in range(n_leaves):
        r, th = coords[i]
        internal_label = leaf_labels[i]  # Internal key (organism_hit_id)
        
        # Extract readable label components from labels_map
        if labels_map and internal_label in labels_map:
            organism, hit_id, ssn_id, accession_id, collapsed_info = labels_map[internal_label]
            # Create display label: Organism + hit gene id
            label_text = f"{organism}_{hit_id}" if hit_id else internal_label
        else:
            label_text = internal_label
        
        color = label_to_cluster_color[internal_label]
        size = 70  # Larger markers
        edge_w = 1.3  # Thicker edges
        
        # Check for highlight
        is_highlight = False
        if labels_map and original_input_sequence_id:
            acc = labels_map[internal_label][3]
            if acc == original_input_sequence_id:
                is_highlight = True
                color = "#DC143C"  # Crimson
                size = 220  # Much larger for prominence
                edge_w = 3.0  # Thicker edge

        # Draw leaf node with publication-quality styling
        ax.scatter([th], [r], color=color, s=size, edgecolors='black', 
                   linewidths=edge_w, zorder=20, alpha=0.9)

        if i not in selected_label_indices:
            continue

        label_th = th
        label_r = r + base_offset + label_levels.get(i, 0) * level_spacing
        label_r = max(label_r, r + 0.015)
        text_len = len(label_text)
        base_th_deg = np.degrees(label_th) % 360

        # With theta_offset=90deg and clockwise direction, visual x-axis maps to theta~90/270.
        d90_leaf = min(abs(base_th_deg - 90.0), 360.0 - abs(base_th_deg - 90.0))
        d270_leaf = min(abs(base_th_deg - 270.0), 360.0 - abs(base_th_deg - 270.0))
        axis_dist_leaf = min(d90_leaf, d270_leaf)
        axis_window_deg = 45.0
        axis_strength = max(0.0, (axis_window_deg - axis_dist_leaf) / axis_window_deg)

        # Keep the default, slightly outside placement for all labels.
        near_x_axis = axis_dist_leaf < x_axis_risk_window_deg
        label_th = th
        va = 'center'
        label_pad = 0.30

        initial_clearance = _clearance_at(label_th, label_r)
        is_long_label = text_len >= 26
        font_size = 16 if not is_highlight else 17
        min_self_clear = _min_self_leaf_clearance(text_len)
        min_self_leaf_px = 8.0 + min(8.0, 0.26 * float(text_len))
        max_inward_depth = _max_inward_depth_for_text(text_len)
        deep_floor = r - (max_inward_depth + (0.060 if is_long_label else 0.020))

        initial_ha = 'left' if (np.degrees(label_th) % 360.0) < 180.0 else 'right'
        initial_overflow_px, initial_leaf_box_dist_px = _probe_bbox_metrics(
            label_th, label_r, label_text, initial_ha, va, font_size, label_pad, th, r
        )

        best_overflow_px = initial_overflow_px
        best_leaf_box_dist_px = initial_leaf_box_dist_px
        best_clearance = initial_clearance
        use_fallback_leader = False

        needs_boundary_adjust = best_overflow_px > 0.0

        if needs_boundary_adjust:
            # First pass: minimal inward radial correction at the original angle.
            for _ in range(10):
                if best_overflow_px <= 0.0 and best_leaf_box_dist_px >= min_self_leaf_px:
                    break

                inward_step = max(
                    0.004,
                    min(0.035, ((best_overflow_px + 1.0) / max(boundary_radius_px, 1.0)) * plot_r_max * 1.8),
                )
                trial_r = max(deep_floor, label_r - inward_step)
                if abs(trial_r - label_r) < 1e-8:
                    break

                trial_clear = _clearance_at(label_th, trial_r)
                trial_self_clear = float(np.linalg.norm(_to_xy(label_th, trial_r) - _to_xy(th, r)))
                if trial_clear < (min_clearance_target - 0.004) or trial_self_clear < (min_self_clear - 0.001):
                    break

                trial_deg = np.degrees(label_th) % 360.0
                trial_ha = 'left' if trial_deg < 180.0 else 'right'
                trial_overflow_px, trial_leaf_box_dist_px = _probe_bbox_metrics(
                    label_th, trial_r, label_text, trial_ha, va, font_size, label_pad, th, r
                )

                improved = (trial_overflow_px < best_overflow_px - 0.2) or (
                    trial_overflow_px <= best_overflow_px + 0.1
                    and trial_leaf_box_dist_px > best_leaf_box_dist_px
                )
                if not improved:
                    break

                label_r = trial_r
                best_overflow_px = trial_overflow_px
                best_leaf_box_dist_px = trial_leaf_box_dist_px
                best_clearance = trial_clear

            # Second pass (only if still outside): tiny angular nudge, keep it close to the leaf.
            if best_overflow_px > 0.0 and near_x_axis:
                theta_step = np.radians(max(0.8, float(label_angle_jitter_deg) * 1.2))
                best_candidate = None
                for k in (0, -1, 1, -2, 2, -3, 3, -4, 4, -5, 5):
                    trial_th = th + k * theta_step
                    trial_r = label_r
                    trial_clear = _clearance_at(trial_th, trial_r)
                    trial_self_clear = float(np.linalg.norm(_to_xy(trial_th, trial_r) - _to_xy(th, r)))
                    if trial_clear < (min_clearance_target - 0.004) or trial_self_clear < (min_self_clear - 0.001):
                        continue

                    trial_deg = np.degrees(trial_th) % 360.0
                    trial_ha = 'left' if trial_deg < 180.0 else 'right'
                    trial_overflow_px, trial_leaf_box_dist_px = _probe_bbox_metrics(
                        trial_th, trial_r, label_text, trial_ha, va, font_size, label_pad, th, r
                    )
                    angular_shift = abs((trial_th - th + np.pi) % (2 * np.pi) - np.pi)
                    score = trial_overflow_px + 20.0 * angular_shift
                    candidate = (score, trial_th, trial_r, trial_overflow_px, trial_leaf_box_dist_px, trial_clear)
                    if best_candidate is None or candidate[0] < best_candidate[0]:
                        best_candidate = candidate

                if best_candidate is not None and best_candidate[3] < best_overflow_px - 0.2:
                    _, label_th, label_r, best_overflow_px, best_leaf_box_dist_px, best_clearance = best_candidate

        # Last resort for dense regions: small outward move and leader, but only when boundary is already safe.
        if best_clearance < min_clearance_target and best_overflow_px <= 0.0:
            fallback_r = min(plot_r_max, max(r + 0.03, label_r + max(0.03, float(leader_min_offset))))
            if fallback_r > label_r + 1e-6:
                label_r = fallback_r
                use_fallback_leader = True

        delta_th = ((label_th - th + np.pi) % (2 * np.pi)) - np.pi
        if delta_th < -np.radians(0.15):
            va = 'bottom'
        elif delta_th > np.radians(0.15):
            va = 'top'

        th_deg = np.degrees(label_th) % 360

        # Keep labels horizontal and anchor by hemisphere for readability.
        rot = 0.0
        ha = 'left' if th_deg < 180.0 else 'right'
        ang_shift = abs((label_th - th + np.pi) % (2 * np.pi) - np.pi)
        # Draw a leader when radial or angular displacement is noticeable.
        leader_needed = ((label_r - r) >= float(leader_min_offset)) or (ang_shift >= np.radians(0.9)) or use_fallback_leader
        if draw_label_leaders and leader_needed:
            # Anchor explicitly at the leaf, then connect toward the label position.
            start_r = r + 0.004
            end_r = max(start_r + 0.004, label_r - 0.008)
            ax.plot([th, label_th], [start_r, end_r],
                    color=(0.35, 0.35, 0.35, 0.62), lw=0.8, zorder=24)
        
        ax.text(label_th, label_r, label_text, 
                fontsize=16 if not is_highlight else 17,
                rotation=rot,
                rotation_mode='anchor',
                ha=ha, 
                va=va, 
                fontweight='bold' if is_highlight else 'normal',
                color="#DC143C" if is_highlight else "black",
                bbox=dict(boxstyle=f'round,pad={label_pad:.2f}', 
                         facecolor='white' if is_highlight else 'white',
                         edgecolor='#DC143C' if is_highlight else '#DDDDDD',
                         alpha=0.9 if is_highlight else 0.85,
                         linewidth=1.4 if is_highlight else 0.6),
                zorder=25)
        placed_label_xy.append(_to_xy(label_th, label_r))

    # Adjust plot limits to accommodate labels pushed further outward
    outer_lim = max_leaf_r + base_offset + (max(0, int(max_stagger_levels)) + 1) * level_spacing + 0.03
    ax.set_ylim(0, plot_r_max)
    
    plt.tight_layout(pad=0.03)

    if out_prefix and save_plots:
        os.makedirs(output_dir, exist_ok=True)
        for fmt in output_formats:
            out_path = os.path.join(output_dir, f"{out_prefix}_circular_tree.{fmt}")
            try:
                fig.savefig(
                    out_path,
                    dpi=dpi,
                    bbox_inches="tight",
                    pad_inches=0.005,
                    facecolor='white',
                    edgecolor='none'
                )
            except PermissionError:
                # On Windows, an open file handle can temporarily lock the target file.
                ts = int(time.time())
                fallback_path = os.path.join(output_dir, f"{out_prefix}_circular_tree_{ts}.{fmt}")
                print(f"[WARN] Could not write {out_path} (permission denied). Saving to {fallback_path} instead.")
                fig.savefig(
                    fallback_path,
                    dpi=dpi,
                    bbox_inches="tight",
                    pad_inches=0.005,
                    facecolor='white',
                    edgecolor='none'
                )
        plt.close(fig)
    elif show:
        plt.show()
    else:
        plt.close(fig)
                       
                

# ----------------------------------------------------------------------
# Dynamic clustering methods
# ----------------------------------------------------------------------
def _find_optimal_clusters_by_gap(linkage_matrix, min_clusters=2, min_gap_ratio=1.5):
    """
    Find optimal number of clusters by detecting large gaps in merge heights.
    Looks for the largest relative jump in the dendrogram.
    
    Parameters:
    - linkage_matrix: scipy linkage matrix
    - min_clusters: minimum number of clusters to consider
    - min_gap_ratio: minimum ratio between consecutive heights to consider it a significant gap
    
    Returns:
    - optimal_distance: distance threshold for cutting the dendrogram
    - num_clusters: estimated number of clusters
    """
    if linkage_matrix.shape[0] < 2:
        return 0.5, 1
    
    # Extract merge heights (distances at which clusters merge)
    heights = linkage_matrix[:, 2]
    
    # Calculate gaps between consecutive merges
    gaps = np.diff(heights)
    gap_ratios = gaps[1:] / (gaps[:-1] + 1e-10)  # Avoid division by zero
    
    # Find the largest gap that produces at least min_clusters
    # Start from the end (highest merges = fewest clusters)
    max_gap_ratio = -np.inf
    best_idx = len(heights) - 1
    
    for i in range(len(heights) - min_clusters, 0, -1):
        if i < len(gap_ratios):
            if gap_ratios[i] > max_gap_ratio and gap_ratios[i] >= min_gap_ratio:
                max_gap_ratio = gap_ratios[i]
                best_idx = i
    
    # Cut at the height just before the large gap
    optimal_distance = (heights[best_idx] + heights[best_idx + 1]) / 2.0 if best_idx < len(heights) - 1 else heights[best_idx] * 1.1
    num_clusters = linkage_matrix.shape[0] - best_idx + 1
    
    return optimal_distance, num_clusters


def _find_optimal_clusters_by_inconsistency(linkage_matrix, depth=2, threshold=1.5):
    """
    Use inconsistency coefficient to find natural clusters.
    Inconsistency measures how different a merge is compared to nearby merges.
    
    Parameters:
    - linkage_matrix: scipy linkage matrix
    - depth: how many levels to look back for comparison
    - threshold: inconsistency threshold for cutting
    
    Returns:
    - cluster_assignments: array of cluster labels
    """
    from scipy.cluster.hierarchy import inconsistent, fcluster
    
    if linkage_matrix.shape[0] < 2:
        return np.ones(linkage_matrix.shape[0] + 1, dtype=int)
    
    # Calculate inconsistency coefficients
    incons = inconsistent(linkage_matrix, d=depth)
    
    # Find a good threshold based on inconsistency statistics
    # Use the mean + threshold * std of inconsistency values
    mean_incons = np.mean(incons[:, 3])  # Column 3 is the inconsistency coefficient
    std_incons = np.std(incons[:, 3])
    
    cutoff = mean_incons + threshold * std_incons
    
    # Use the inconsistency criterion for clustering
    cluster_assignments = fcluster(linkage_matrix, cutoff, criterion='inconsistent', depth=depth)
    
    return cluster_assignments


def _find_optimal_clusters_by_elbow(linkage_matrix, max_clusters=None):
    """
    Use the elbow method on within-cluster distances.
    Finds the point where adding more clusters doesn't significantly improve separation.
    
    Parameters:
    - linkage_matrix: scipy linkage matrix
    - max_clusters: maximum number of clusters to test (default: sqrt(n))
    
    Returns:
    - optimal_distance: distance threshold for cutting
    - optimal_k: optimal number of clusters
    """
    from scipy.cluster.hierarchy import fcluster
    
    n = linkage_matrix.shape[0] + 1  # Number of leaves
    
    if n < 3:
        return linkage_matrix[-1, 2] / 2.0 if linkage_matrix.shape[0] > 0 else 0.5, 1
    
    if max_clusters is None:
        max_clusters = min(int(np.sqrt(n)), n - 1)
    
    max_clusters = max(2, min(max_clusters, n - 1))
    
    # Test different numbers of clusters
    wcss_values = []  # Within-cluster sum of squares (approximated)
    k_range = range(1, max_clusters + 1)
    
    heights = linkage_matrix[:, 2]
    
    for k in k_range:
        # Height at which we'd get k clusters
        if k == 1:
            wcss = heights[-1] if len(heights) > 0 else 0
        else:
            # The height where we cut to get k clusters
            idx = -(k - 1) if k <= len(heights) else 0
            wcss = heights[idx] if idx < len(heights) else 0
        
        wcss_values.append(wcss)
    
    wcss_values = np.array(wcss_values)
    
    # Find elbow using the method of maximum distance to the line
    if len(wcss_values) < 3:
        return heights[-1] / 2.0, 2
    
    # Normalize
    x = np.arange(len(wcss_values))
    y = wcss_values
    
    # Line from first to last point
    p1 = np.array([x[0], y[0]])
    p2 = np.array([x[-1], y[-1]])
    
    # Calculate distances from each point to the line
    distances = np.zeros(len(x))
    for i in range(len(x)):
        p = np.array([x[i], y[i]])
        distances[i] = np.abs(np.cross(p2 - p1, p1 - p)) / np.linalg.norm(p2 - p1)
    
    # Find the elbow (maximum distance)
    optimal_idx = np.argmax(distances)
    optimal_k = k_range[optimal_idx]
    
    # Find the distance threshold for optimal_k clusters
    if optimal_k == 1:
        optimal_distance = heights[-1] * 1.1 if len(heights) > 0 else 1.0
    else:
        cut_idx = -(optimal_k - 1)
        if cut_idx >= -len(heights):
            optimal_distance = (heights[cut_idx] + heights[cut_idx - 1]) / 2.0 if cut_idx > -len(heights) else heights[cut_idx]
        else:
            optimal_distance = heights[0] / 2.0
    
    return optimal_distance, optimal_k


def _find_optimal_clusters_by_lifetime(linkage_matrix, percentile=90):
    """
    Use cluster lifetime (persistence) to find stable clusters.
    Clusters that exist for a long 'time' (distance range) are considered robust.
    
    Parameters:
    - linkage_matrix: scipy linkage matrix
    - percentile: percentile of lifetimes to use as threshold
    
    Returns:
    - optimal_distance: distance threshold
    - num_clusters: estimated number of clusters
    """
    if linkage_matrix.shape[0] < 2:
        return 0.5, 1
    
    heights = linkage_matrix[:, 2]
    n = linkage_matrix.shape[0] + 1
    
    # Calculate lifetimes: for each merge, how long did the clusters exist?
    # Lifetime = height at which cluster dies - height at which it was born
    lifetimes = []
    birth_times = np.zeros(2 * n - 1)  # Birth time for each node
    
    for i in range(len(heights)):
        c1, c2 = int(linkage_matrix[i, 0]), int(linkage_matrix[i, 1])
        merge_height = heights[i]
        new_cluster = n + i
        
        # Lifetime of clusters being merged
        life1 = merge_height - birth_times[c1]
        life2 = merge_height - birth_times[c2]
        
        lifetimes.append(life1)
        lifetimes.append(life2)
        
        # New cluster born at this height
        birth_times[new_cluster] = merge_height
    
    lifetimes = np.array(lifetimes)
    
    # Use percentile of lifetimes as threshold
    significant_lifetime = np.percentile(lifetimes, percentile)
    
    # Find merges where both clusters have long lifetimes
    cut_height = None
    for i in range(len(heights)):
        c1, c2 = int(linkage_matrix[i, 0]), int(linkage_matrix[i, 1])
        merge_height = heights[i]
        
        life1 = merge_height - birth_times[c1]
        life2 = merge_height - birth_times[c2]
        
        if life1 >= significant_lifetime or life2 >= significant_lifetime:
            cut_height = merge_height
            break
    
    if cut_height is None:
        cut_height = np.median(heights)
    
    # Estimate number of clusters at this cut
    num_clusters = np.sum(heights > cut_height) + 1
    
    return cut_height, num_clusters


def _find_optimal_clusters_by_topology(linkage_matrix, max_clusters=None, coherence_threshold=0.7):
    """
    Find clusters based on tree topology to ensure visual coherence.
    Groups leaves that share recent common ancestors and form complete subtrees.
    
    Parameters:
    - linkage_matrix: scipy linkage matrix
    - max_clusters: maximum number of clusters (default: sqrt(n))
    - coherence_threshold: how coherent subtrees should be (0-1, higher = more coherent)
    
    Returns:
    - optimal_distance: distance threshold
    - num_clusters: number of clusters
    """
    from scipy.cluster.hierarchy import fcluster
    
    n = linkage_matrix.shape[0] + 1  # Number of leaves
    
    if n < 3:
        return linkage_matrix[-1, 2] / 2.0 if linkage_matrix.shape[0] > 0 else 0.5, 1
    
    if max_clusters is None:
        max_clusters = max(2, min(int(np.sqrt(n) * 1.5), n - 1))
    
    heights = linkage_matrix[:, 2]
    
    # Build tree structure
    children_dict = {}
    for i in range(linkage_matrix.shape[0]):
        node_id = n + i
        children_dict[node_id] = (int(linkage_matrix[i, 0]), int(linkage_matrix[i, 1]))
    
    def get_descendants(node_id):
        """Get all leaf descendants of a node"""
        if node_id < n:
            return [node_id]
        c1, c2 = children_dict[node_id]
        return get_descendants(c1) + get_descendants(c2)
    
    # Evaluate different cut heights for coherence
    best_score = -np.inf
    best_height = heights[-1] / 2.0
    best_k = 2
    
    # Test different numbers of clusters
    for k in range(2, max_clusters + 1):
        # Cut tree to get k clusters
        if k > len(heights):
            continue
            
        cut_idx = -(k - 1)
        if cut_idx < -len(heights):
            continue
            
        cut_height = (heights[cut_idx] + heights[cut_idx - 1]) / 2.0 if cut_idx > -len(heights) else heights[cut_idx]
        
        # Get cluster assignments
        assignments = fcluster(linkage_matrix, cut_height, criterion='distance')
        
        # Calculate coherence score: how well do clusters correspond to subtrees?
        # Look at the tree structure at this cut height
        coherence_scores = []
        
        for cluster_id in np.unique(assignments):
            cluster_leaves = np.where(assignments == cluster_id)[0]
            if len(cluster_leaves) < 2:
                coherence_scores.append(1.0)  # Single leaf is perfectly coherent
                continue
            
            # Find the lowest common ancestor (LCA) height for leaves in this cluster
            # A good cluster should have leaves that diverge at similar heights
            cluster_heights = []
            for i, leaf1 in enumerate(cluster_leaves):
                for leaf2 in cluster_leaves[i+1:]:
                    # Find LCA height in linkage matrix
                    # This is approximate - look for merges involving these leaves
                    cluster_heights.append(0.0)  # Simplified for now
            
            if cluster_heights:
                # Coherence is high if all leaves in cluster are close in tree
                coherence = 1.0 - np.std(cluster_heights) / (np.mean(heights) + 1e-10)
            else:
                coherence = 1.0
            
            coherence_scores.append(coherence)
        
        # Score combines number of clusters with coherence
        avg_coherence = np.mean(coherence_scores)
        
        # Penalize too few or too many clusters, reward high coherence
        k_penalty = abs(k - np.sqrt(n)) / n
        score = avg_coherence * (1.0 - k_penalty * 0.5)
        
        if score > best_score:
            best_score = score
            best_height = cut_height
            best_k = k
    
    return best_height, best_k


def _find_optimal_clusters_by_maxclust(linkage_matrix, target_clusters=None):
    """
    Find optimal clustering by specifying target number of clusters.
    Uses the maxclust criterion to create exactly k clusters.
    
    Parameters:
    - linkage_matrix: scipy linkage matrix
    - target_clusters: desired number of clusters (default: sqrt(n))
    
    Returns:
    - cluster_assignments: array of cluster labels
    - num_clusters: actual number of clusters created
    """
    from scipy.cluster.hierarchy import fcluster
    
    n = linkage_matrix.shape[0] + 1
    
    if n < 2:
        return np.ones(n, dtype=int), 1
    
    if target_clusters is None:
        target_clusters = max(2, int(np.sqrt(n)))
    
    target_clusters = max(1, min(target_clusters, n))
    
    # Use maxclust criterion for exact number of clusters
    cluster_assignments = fcluster(linkage_matrix, target_clusters, criterion='maxclust')
    
    return cluster_assignments, target_clusters


def determine_optimal_clustering(linkage_matrix, method='gap', **kwargs):
    """
    Determine optimal clustering using various dynamic methods.
    
    Parameters:
    - linkage_matrix: scipy linkage matrix
    - method: 'gap', 'inconsistency', 'elbow', 'lifetime', 'topology', 'maxclust', or 'combined'
    - **kwargs: additional parameters for specific methods
    
    Returns:
    - cluster_assignments: array of cluster labels
    - info: dict with method-specific information
    """
    from scipy.cluster.hierarchy import fcluster
    
    n = linkage_matrix.shape[0] + 1
    
    if n < 2:
        return np.ones(n, dtype=int), {'method': method, 'n_clusters': 1}
    
    info = {'method': method}
    
    if method == 'gap':
        distance_threshold, n_clusters = _find_optimal_clusters_by_gap(
            linkage_matrix, 
            min_clusters=kwargs.get('min_clusters', 2),
            min_gap_ratio=kwargs.get('min_gap_ratio', 1.5)
        )
        info['distance_threshold'] = distance_threshold
        info['n_clusters'] = n_clusters
        cluster_assignments = fcluster(linkage_matrix, distance_threshold, criterion='distance')
        
    elif method == 'inconsistency':
        cluster_assignments = _find_optimal_clusters_by_inconsistency(
            linkage_matrix,
            depth=kwargs.get('depth', 2),
            threshold=kwargs.get('threshold', 1.5)
        )
        info['n_clusters'] = len(np.unique(cluster_assignments))
        
    elif method == 'elbow':
        distance_threshold, n_clusters = _find_optimal_clusters_by_elbow(
            linkage_matrix,
            max_clusters=kwargs.get('max_clusters', None)
        )
        info['distance_threshold'] = distance_threshold
        info['n_clusters'] = n_clusters
        cluster_assignments = fcluster(linkage_matrix, distance_threshold, criterion='distance')
        
    elif method == 'lifetime':
        distance_threshold, n_clusters = _find_optimal_clusters_by_lifetime(
            linkage_matrix,
            percentile=kwargs.get('percentile', 90)
        )
        info['distance_threshold'] = distance_threshold
        info['n_clusters'] = n_clusters
        cluster_assignments = fcluster(linkage_matrix, distance_threshold, criterion='distance')
        
    elif method == 'topology':
        distance_threshold, n_clusters = _find_optimal_clusters_by_topology(
            linkage_matrix,
            max_clusters=kwargs.get('max_clusters', None),
            coherence_threshold=kwargs.get('coherence_threshold', 0.7)
        )
        info['distance_threshold'] = distance_threshold
        info['n_clusters'] = n_clusters
        info['note'] = 'Tree-topology aware clustering for visual coherence'
        cluster_assignments = fcluster(linkage_matrix, distance_threshold, criterion='distance')
        
    elif method == 'maxclust':
        cluster_assignments, n_clusters = _find_optimal_clusters_by_maxclust(
            linkage_matrix,
            target_clusters=kwargs.get('target_clusters', None)
        )
        info['n_clusters'] = n_clusters
        info['note'] = f'Forced {n_clusters} clusters using maxclust criterion'
        
    elif method == 'combined':
        # Use multiple methods and take consensus
        methods_to_try = ['gap', 'elbow', 'topology']
        all_n_clusters = []
        all_assignments = []
        
        for m in methods_to_try:
            try:
                assignments, m_info = determine_optimal_clustering(linkage_matrix, method=m)
                all_n_clusters.append(m_info['n_clusters'])
                all_assignments.append(assignments)
            except:
                pass
        
        if all_n_clusters:
            # Use median number of clusters
            median_k = int(np.median(all_n_clusters))
            info['n_clusters'] = median_k
            info['individual_estimates'] = all_n_clusters
            
            # Find the assignment closest to median
            differences = [abs(len(np.unique(a)) - median_k) for a in all_assignments]
            best_idx = np.argmin(differences)
            cluster_assignments = all_assignments[best_idx]
        else:
            # Fallback
            cluster_assignments = fcluster(linkage_matrix, 0.7, criterion='distance')
            info['n_clusters'] = len(np.unique(cluster_assignments))
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return cluster_assignments, info


# ----------------------------------------------------------------------
# Main clustering function (using circular tree plotting)
# ----------------------------------------------------------------------
def cluster_gene_neighborhoods_from_sqlite(
    db_path,
    genes_table=GENES_TABLE,
    neighbors_table=NEIGHBORS_TABLE,
    col_neighborhood_id=COL_NEIGHBORHOOD_ID,
    col_gene_id=COL_GENE_ID,
    col_linking_key=COL_LINKING_KEY,
    col_accession_id=COL_ACCESSION_ID,
    col_function_desc=COL_FUNCTION_DESC,
    col_pfam_ids=COL_PFAM_IDS,
    col_interpro_ids=COL_INTERPRO_IDS,
    col_rel_start=COL_REL_START,
    col_rel_stop=COL_REL_STOP,
    col_ssn_cluster_id=COL_SSN_CLUSTER_ID,
    hit_gene_weight_factor=HIT_GENE_WEIGHT_FACTOR,
    direct_neighbor_weight_factor=DIRECT_NEIGHBOR_WEIGHT_FACTOR,
    differentiate_by_ssn_cluster=True,
    ssn_cluster_value_to_filter=DEFAULT_SSN_CLUSTER_VALUE_TO_FILTER,
    collapse_identical_neighborhoods=COLLAPSE_IDENTICAL_NEIGHBORHOODS,
    collapse_core_similarity_threshold=COLLAPSE_CORE_SIMILARITY_THRESHOLD,
    collapse_full_neighborhood_similarity_threshold=COLLAPSE_FULL_NEIGHBORHOOD_SIMILARITY_THRESHOLD,
    max_neighbors_per_side=MAX_NEIGHBORS_PER_SIDE,
    original_input_sequence_id=None,
    distance_threshold=0.8,
    clustering_method='gap',
    clustering_params=None,
    plot_max_labels=28,
    plot_min_labels_per_cluster=1,
    plot_extra_labels_per_cluster=1,
    plot_min_label_separation_deg=40.0,
    plot_min_label_angle_deg=6.0,
    plot_max_stagger_levels=4,
    plot_label_base_offset=0.035,
    plot_label_stagger_step=0.02,
    plot_draw_label_leaders=True,
    plot_leader_min_offset=0.05,
    csv_output_path=None,
    save_plots=SAVE_PLOTS,
    output_dir=OUTPUT_DIR,
    output_formats=OUTPUT_FORMATS,
    dpi=DPI,
    report_file_handle=None,
    parallelize_pdist=False,
):
    """
    Cluster gene neighborhoods based on domain/annotation features and
    draw circular phylograms for each SSN cluster (or all neighborhoods).
    
    Clustering methods:
    - 'gap': Finds large gaps in merge heights (fast, good for well-separated clusters)
    - 'inconsistency': Uses inconsistency coefficients (hierarchical structure aware)
    - 'elbow': Elbow method on within-cluster distances (classic approach)
    - 'lifetime': Cluster persistence/lifetime analysis (finds stable clusters)
    - 'topology': Tree-topology aware (best for visual coherence) **RECOMMENDED**
    - 'maxclust': Forces specific number of clusters (when you know k)
    - 'combined': Consensus of multiple methods (most robust but slower)
    - 'static': Use fixed distance_threshold (legacy)
    """
    def log(msg):
        print(msg)
        if report_file_handle:
            report_file_handle.write(msg + "\n")

    t0_all = time.time()
    conn = sqlite3.connect(db_path)

    # --- Fetch data ---
    log("Fetching hit gene data ...")
    q_hits = f"""
        SELECT {col_gene_id}, {col_neighborhood_id}, {col_function_desc},
               {col_pfam_ids}, {col_interpro_ids}, {col_ssn_cluster_id},
               {col_accession_id}
        FROM {genes_table}
    """
    hit_genes_df = pd.read_sql_query(q_hits, conn)
    log(f"  Retrieved {len(hit_genes_df)} hit genes")

    if hit_genes_df.empty:
        log("No hit genes found; aborting.")
        conn.close()
        return {}, {}, {}

    log("Fetching neighbor data ...")
    q_neighbors = f"""
        SELECT {col_linking_key}, {col_gene_id}, {col_function_desc},
               {col_pfam_ids}, {col_interpro_ids},
               {col_rel_start}, {col_rel_stop}
        FROM {neighbors_table}
    """
    neighbors_df = pd.read_sql_query(q_neighbors, conn)
    conn.close()
    log(f"  Retrieved {len(neighbors_df)} neighbor rows")

    # Group neighbors by linking key once
    log("Grouping neighbors by hit gene id ...")
    neighbors_by_link_id = {
        key: group.to_dict("records")
        for key, group in tqdm(
            neighbors_df.groupby(col_linking_key),
            desc="  Grouping neighbors",
            leave=False,
        )
    }
    del neighbors_df
    gc.collect()

    all_neighborhood_features = defaultdict(set)
    core_neighborhood_features = defaultdict(set)
    full_neighborhood_labels_map = {}
    raw_ssn_counts = defaultdict(int)
    trimmed_neighborhood_count = 0
    trimmed_neighbor_total = 0

    # --- Feature extraction ---
    log(f"Extracting features for {len(hit_genes_df)} neighborhoods ...")
    for _, hit_row in tqdm(
        hit_genes_df.iterrows(),
        total=len(hit_genes_df),
        desc="  Neighborhoods",
        unit="nh",
    ):
        hit_id = hit_row[col_gene_id]
        organism = hit_row[col_neighborhood_id]
        ssn_id = hit_row[col_ssn_cluster_id]
        accession_id = hit_row[col_accession_id]

        raw_ssn_counts[ssn_id] += 1
        nh_label = f"{organism}_{hit_id}"

        full_feats = set()
        core_feats = set()

        # hit gene, strongly weighted for full neighborhood features
        hit_full = extract_features_from_gene_row(
            hit_row,
            current_weight_factor=hit_gene_weight_factor,
            base_prefix="HIT_",
            include_desc=True,
            include_pfam=True,
            include_interpro=True,
        )
        full_feats.update(hit_full)

        hit_core = extract_features_from_gene_row(
            hit_row,
            current_weight_factor=1,
            base_prefix="HIT_CORE_",
            include_desc=False,
            include_pfam=True,
            include_interpro=True,
        )
        core_feats.update(hit_core)

        full_neighborhood_labels_map[nh_label] = (
            organism,
            hit_id,
            ssn_id,
            accession_id,
            None,
        )

        # Keep only the nearest neighbors per side (left/right) around the hit gene.
        raw_neighbor_rows = neighbors_by_link_id.get(hit_id, [])
        neighbor_rows = _limit_neighbors_per_side(
            raw_neighbor_rows,
            col_rel_start=col_rel_start,
            col_rel_stop=col_rel_stop,
            max_neighbors_per_side=max_neighbors_per_side,
        )
        if len(neighbor_rows) < len(raw_neighbor_rows):
            trimmed_neighborhood_count += 1
            trimmed_neighbor_total += len(raw_neighbor_rows) - len(neighbor_rows)

        # determine closest left/right neighbors by relative coordinates
        closest_left_id = None
        closest_right_id = None
        max_neg_rel_stop = -np.inf
        min_pos_rel_start = np.inf

        for nrow in neighbor_rows:
            rel_start = nrow[col_rel_start]
            rel_stop = nrow[col_rel_stop]
            nid = nrow[col_gene_id]

            if rel_stop is not None and rel_stop < 0 and rel_stop > max_neg_rel_stop:
                max_neg_rel_stop = rel_stop
                closest_left_id = nid
            if rel_start is not None and rel_start > 0 and rel_start < min_pos_rel_start:
                min_pos_rel_start = rel_start
                closest_right_id = nid

        for nrow in neighbor_rows:
            nid = nrow[col_gene_id]
            is_direct = (
                (closest_left_id is not None and nid == closest_left_id)
                or (closest_right_id is not None and nid == closest_right_id)
            )
            w = direct_neighbor_weight_factor if is_direct else 1

            nf_full = extract_features_from_gene_row(
                nrow,
                current_weight_factor=w,
                base_prefix="N_",
                include_desc=True,
                include_pfam=True,
                include_interpro=True,
            )
            full_feats.update(nf_full)

            if is_direct:
                nf_core = extract_features_from_gene_row(
                    nrow,
                    current_weight_factor=1,
                    base_prefix="N_CORE_",
                    include_desc=False,
                    include_pfam=True,
                    include_interpro=True,
                )
                core_feats.update(nf_core)

        all_neighborhood_features[nh_label] = full_feats
        core_neighborhood_features[nh_label] = core_feats

    del neighbors_by_link_id
    gc.collect()
    log("Feature extraction complete.")
    if max_neighbors_per_side is not None and max_neighbors_per_side > 0:
        log(
            f"Applied side-aware neighbor cap (+/-{max_neighbors_per_side}): removed "
            f"{trimmed_neighbor_total} outer neighbors across {trimmed_neighborhood_count} neighborhoods."
        )

    log("\nRaw SSN id distribution:")
    for sid, count in sorted(raw_ssn_counts.items(), key=lambda x: str(x[0])):
        log(f"  SSN {sid}: {count} neighborhoods")
    log("-" * 60)

    if not all_neighborhood_features:
        log("No neighborhoods with features; aborting.")
        return {}, {}, {}

    # --- Collapsing similar neighborhoods (optional) ---
    if collapse_identical_neighborhoods:
        final_neighborhood_features, final_neighborhood_labels_map, collapsed_groups_report = _perform_collapsing(
            all_neighborhood_features,
            full_neighborhood_labels_map,
            core_neighborhood_features,
            collapse_core_similarity_threshold,
            collapse_full_neighborhood_similarity_threshold,
            output_prefix="[Collapsing]",
            report_file=report_file_handle,
            parallelize_pdist=parallelize_pdist,
        )
    else:
        log("Collapsing disabled; using all neighborhoods as-is.")
        final_neighborhood_features = all_neighborhood_features
        final_neighborhood_labels_map = full_neighborhood_labels_map
        collapsed_groups_report = {}

    del all_neighborhood_features
    del core_neighborhood_features
    gc.collect()

    # sanity check on features
    if not final_neighborhood_features:
        log("No final neighborhoods after collapsing; aborting.")
        return {}, {}, collapsed_groups_report

    # determine grouping (SSN-separated or all)
    ssn_groups = defaultdict(list)
    if differentiate_by_ssn_cluster:
        for nh_label, (_, _, ssn_id, _, _) in final_neighborhood_labels_map.items():
            if ssn_id not in ssn_cluster_value_to_filter:
                ssn_groups[ssn_id].append(nh_label)
        log(f"\nProcessing {len(ssn_groups)} SSN groups (after filtering).")
        if ssn_groups:
            log("  SSN ids: " + ", ".join(map(str, sorted(ssn_groups.keys(), key=str))))
    else:
        ssn_groups["All_Neighborhoods"] = sorted(final_neighborhood_features.keys())
        log("Processing all neighborhoods together (no SSN separation).")

    clusters_output = defaultdict(dict)
    clustering_csv_rows = []
    if differentiate_by_ssn_cluster and not ssn_groups:
        log("No valid SSN clusters with neighborhoods; nothing to cluster.")
        return {}, final_neighborhood_labels_map, collapsed_groups_report

    # --- Per-group clustering and plotting ---
    for ssn_id, nh_labels in tqdm(
        ssn_groups.items(),
        desc="Clustering SSN groups",
        unit="group",
    ):
        group_start = time.time()
        label_list = sorted(nh_labels)
        n_nh = len(label_list)
        if n_nh < 2:
            log(f"SSN {ssn_id}: only {n_nh} neighborhood(s); skipping clustering.")
            clusters_output[ssn_id] = {1: label_list}
            for nh in label_list:
                org, hit_id, ssn_val, acc, collapsed_info = final_neighborhood_labels_map.get(
                    nh, ("UNKNOWN", "UNKNOWN", None, "UNKNOWN", None)
                )
                collapsed_count, collapsed_code = (collapsed_info if collapsed_info else (1, ""))
                clustering_csv_rows.append({
                    'ssn_id': ssn_val,
                    'neighborhood_label': nh,
                    'organism': org,
                    'hit_id': hit_id,
                    'accession_id': acc,
                    'cluster_id': 1,
                    'cluster_size': len(label_list),
                    'clustering_method': clustering_method,
                    'distance_threshold_used': np.nan,
                    'silhouette_like_score': np.nan,
                    'radial_distance': np.nan,
                    'angular_degree': np.nan,
                    'radial_rank_outward': np.nan,
                    'is_original_input': bool(original_input_sequence_id and acc == original_input_sequence_id),
                    'collapsed_count': collapsed_count,
                    'collapsed_code': collapsed_code,
                })
            continue
        
        print(f"Processing SSN {ssn_id} with {len(label_list)} neighborhoods.")
        
        log(f"\n--- SSN {ssn_id}: {n_nh} neighborhoods ---")
        group_feats = {l: final_neighborhood_features[l] for l in label_list}
        group_labels_map = {l: final_neighborhood_labels_map[l] for l in label_list}

        # vocabulary and binary matrix
        vocab = sorted(set.union(*group_feats.values()))
        if not vocab:
            log(f"  SSN {ssn_id}: no features; skipping.")
            clusters_output[ssn_id] = {1: label_list}
            continue

        ft_idx = {f: i for i, f in enumerate(vocab)}
        mat_lil = sp.lil_matrix((n_nh, len(vocab)), dtype=np.int8)
        for i, nh in enumerate(tqdm(label_list, desc=f"  Features SSN {ssn_id}", leave=False)):
            for f in group_feats[nh]:
                j = ft_idx[f]
                mat_lil[i, j] = 1
        mat = mat_lil.tocsr()
        del mat_lil
        gc.collect()

        # all identical?
        if n_nh > 1 and all((mat[0] != mat[i]).nnz == 0 for i in range(1, n_nh)):
            log(f"  SSN {ssn_id}: all neighborhoods identical; no tree/distance.")
            clusters_output[ssn_id] = {1: label_list}
            del mat
            gc.collect()
            continue

        # distances + linkage
        log(f"  SSN {ssn_id}: computing Jaccard distances ...")
        t0 = time.time()
        dists = parallel_pdist_jaccard(mat, num_cores=-1 if parallelize_pdist else 1)
        log(f"    distance calc in {time.time() - t0:.2f}s")

        log(f"  SSN {ssn_id}: linkage ...")
        t0 = time.time()
        linked = linkage(dists, method="average")
        log(f"    linkage in {time.time() - t0:.2f}s")

        del dists
        gc.collect()

        # Dynamic cluster assignments
        log(f"  SSN {ssn_id}: determining optimal clusters (method: {clustering_method}) ...")
        t0 = time.time()
        
        if clustering_method == 'static':
            # Legacy: use fixed threshold
            cluster_assignments = fcluster(linked, distance_threshold, criterion="distance")
            n_clusters = len(np.unique(cluster_assignments))
            used_distance_threshold = distance_threshold
            log(f"    static threshold={distance_threshold:.3f} -> {n_clusters} clusters")
        else:
            # Dynamic clustering
            params = clustering_params or {}
            cluster_assignments, cluster_info = determine_optimal_clustering(
                linked, method=clustering_method, **params
            )
            n_clusters = cluster_info['n_clusters']
            used_distance_threshold = cluster_info.get('distance_threshold', np.nan)
            log(f"    {clustering_method} method -> {n_clusters} clusters")
            if 'distance_threshold' in cluster_info:
                log(f"    dynamic threshold: {cluster_info['distance_threshold']:.3f}")
            if 'individual_estimates' in cluster_info:
                log(f"    individual method estimates: {cluster_info['individual_estimates']}")
        
        log(f"    clustering complete in {time.time() - t0:.2f}s")
        
        clusters = defaultdict(list)
        for i, cid in enumerate(cluster_assignments):
            clusters[cid].append(label_list[i])
        clusters_output[ssn_id] = clusters
        cluster_sizes = {cid: len(members) for cid, members in clusters.items()}
        # Per-item export metrics
        silhouette_like_scores = _compute_silhouette_like_scores_from_sparse_binary(
            mat, cluster_assignments, max_n=300
        )
        radii_for_csv = _compute_dual_scale_radii(
            mat,
            linked,
            label_list,
            inner_boundary=0.45,
            consensus_threshold=0.2,
            leaf_stretch_power=7,
        )
        _, subtree_weight_for_csv, _ = _compute_tree_metrics(linked, label_list)
        angles_for_csv = _compute_cluster_aware_angles(
            linked, label_list, subtree_weight_for_csv, gap_factor=0.1
        )
        leaf_radii = np.array([radii_for_csv[i] for i in range(n_nh)], dtype=float)
        order_outward = np.argsort(-leaf_radii)
        radial_rank = {int(idx): int(rank + 1) for rank, idx in enumerate(order_outward)}
        for i, nh in enumerate(label_list):
            cid = int(cluster_assignments[i])
            org, hit_id, ssn_val, acc, collapsed_info = group_labels_map.get(
                nh, ("UNKNOWN", "UNKNOWN", None, "UNKNOWN", None)
            )
            collapsed_count, collapsed_code = (collapsed_info if collapsed_info else (1, ""))
            clustering_csv_rows.append({
                'ssn_id': ssn_val,
                'neighborhood_label': nh,
                'organism': org,
                'hit_id': hit_id,
                'accession_id': acc,
                'cluster_id': cid,
                'cluster_size': int(cluster_sizes.get(cid, 0)),
                'clustering_method': clustering_method,
                'distance_threshold_used': float(used_distance_threshold) if pd.notna(used_distance_threshold) else np.nan,
                'silhouette_like_score': float(silhouette_like_scores[i]) if pd.notna(silhouette_like_scores[i]) else np.nan,
                'radial_distance': float(radii_for_csv[i]),
                'angular_degree': float(np.degrees(angles_for_csv[i]) % 360.0),
                'radial_rank_outward': int(radial_rank[i]),
                'is_original_input': bool(original_input_sequence_id and acc == original_input_sequence_id),
                'collapsed_count': collapsed_count,
                'collapsed_code': collapsed_code,
            })

        # prepare coloring
        _, label_to_color = _collect_cluster_colors(cluster_assignments, label_list, cmap_name="tab20")

        # draw circular tree
        title = f"SSN {ssn_id} gene neighborhoods"
        out_prefix = f"SSN_{ssn_id}_gnn"
        _draw_circular_tree_with_clusters(
            linkage_matrix=linked,
            leaf_labels=label_list,
            cluster_assignments=cluster_assignments,
            label_to_cluster_color=label_to_color,
            original_input_sequence_id=original_input_sequence_id,
            labels_map=group_labels_map,
            title=title,
            out_prefix=out_prefix,
            output_dir=output_dir,
            output_formats=output_formats,
            dpi=dpi,
            show=not save_plots,
            save_plots=save_plots,
            feature_matrix=mat,
            inner_boundary=0.45,
            consensus_threshold=0.2,
            leaf_stretch_power=7,
            gap_factor=0.1,
            max_labels=plot_max_labels,
            min_labels_per_cluster=plot_min_labels_per_cluster,
            extra_labels_per_cluster=plot_extra_labels_per_cluster,
            min_label_separation_deg=plot_min_label_separation_deg,
            min_label_angle_deg=plot_min_label_angle_deg,
            max_stagger_levels=plot_max_stagger_levels,
            label_base_offset=plot_label_base_offset,
            label_stagger_step=plot_label_stagger_step,
            draw_label_leaders=plot_draw_label_leaders,
            leader_min_offset=plot_leader_min_offset,
        )

        log(f"--- SSN {ssn_id} done in {time.time() - group_start:.2f}s ---")
        del linked
        gc.collect()

    if csv_output_path:
        csv_dir = os.path.dirname(csv_output_path)
        if csv_dir:
            os.makedirs(csv_dir, exist_ok=True)
        csv_df = pd.DataFrame(clustering_csv_rows)
        try:
            csv_df.to_csv(csv_output_path, index=False)
            log(f"Detailed clustering table written to: {csv_output_path} ({len(clustering_csv_rows)} rows)")
        except PermissionError:
            ts = int(time.time())
            root, ext = os.path.splitext(csv_output_path)
            fallback_csv = f"{root}_{ts}{ext or '.csv'}"
            log(f"[WARN] Could not write {csv_output_path} (permission denied). Writing to {fallback_csv} instead.")
            csv_df.to_csv(fallback_csv, index=False)
            log(f"Detailed clustering table written to: {fallback_csv} ({len(clustering_csv_rows)} rows)")

    log(f"\nTotal runtime: {time.time() - t0_all:.2f}s")
    return clusters_output, final_neighborhood_labels_map, collapsed_groups_report

In [2]:
# ----------------------------------------------------------------------
# High-level driver / example usage
# ----------------------------------------------------------------------
SQLITE_DB_PATH = r"D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\CaMES\CaMES_10kBlast_10e_50eEdge_noFilter_300AST_min900AA_withoutEgtD_withoutMethyltrans_10N\39061_CaMES_10kBlast_10e_50eEdge_noFilter_300AST_min900AA_withoutEgtD_withoutMethyltrans_10N.sqlite"
ORIGINAL_INPUT_SEQUENCE_ID = "A0A7V4WV16"  # or None
DIFFERENTIATE_BY_SSN_CLUSTER = False  # Whether to cluster neighborhoods separately by SSN cluster

# --- Clustering method configuration ---
# Choose: 'gap', 'inconsistency', 'elbow', 'lifetime', 'topology', 'maxclust', 'combined', or 'static'
# CLUSTERING_METHOD = 'topology'  # RECOMMENDED: ensures visually coherent clusters matching tree structure
# CLUSTERING_PARAMS = {
#     'max_clusters': None,        # Maximum clusters (None = auto, uses sqrt(n))
#     'coherence_threshold': 0.7   # How coherent subtrees should be (0-1, higher = more coherent)
# }

# Alternative methods:
# CLUSTERING_METHOD = 'gap'  # Fast, good for well-separated clusters
# CLUSTERING_PARAMS = {'min_clusters': 2, 'min_gap_ratio': 1.5}

# CLUSTERING_METHOD = 'maxclust'  # Force specific number of clusters
# CLUSTERING_PARAMS = {'target_clusters': 5}

CLUSTERING_METHOD = 'combined'  # Most robust (consensus of multiple methods)
CLUSTERING_PARAMS = {}  # Optional per-method params; keep empty for defaults

# For legacy behavior with fixed threshold:
# CLUSTERING_METHOD = 'static'

chosen_distance_threshold = 0.6  # Only used if CLUSTERING_METHOD = 'static'
PARALLELIZE_PDIST_ENABLED = True
MAX_NEIGHBORS_PER_SIDE_ACTIVE = 10  # Set to 5 for a tighter window, or None to disable trimming

# Label density controls for publication output.
PLOT_MAX_LABELS = 26  # Global upper bound of labels per tree
PLOT_MIN_LABELS_PER_CLUSTER = 1  # Always annotate at least one representative per cluster
PLOT_EXTRA_LABELS_PER_CLUSTER = 1  # Add additional peripheral labels per cluster when possible
PLOT_MIN_LABEL_SEPARATION_DEG = 40.0  # Enforce angular spacing between non-mandatory labels
PLOT_MIN_LABEL_ANGLE_DEG = 6.5  # Higher values increase anti-overlap behavior
PLOT_MAX_STAGGER_LEVELS = 4  # Radial staggering levels for dense angular regions
PLOT_LABEL_BASE_OFFSET = 0.035  # Keep labels close to leaf nodes
PLOT_LABEL_STAGGER_STEP = 0.02  # Small radial offset for local collisions
PLOT_DRAW_LABEL_LEADERS = True  # Draw short leader lines only when needed
PLOT_LEADER_MIN_OFFSET = 0.05  # Start drawing leaders if label moved at least this far

COLLAPSE_IDENTICAL_NEIGHBORHOODS_ACTIVE = True
COLLAPSE_CORE_SIMILARITY_THRESHOLD_ACTIVE = 0.0
COLLAPSE_FULL_NEIGHBORHOOD_SIMILARITY_THRESHOLD_ACTIVE = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)
report_suffix = "_ssn_differentiated" if DIFFERENTIATE_BY_SSN_CLUSTER else "_all_neighborhoods"
report_filename = f"{REPORT_FILENAME_BASE}{report_suffix}.txt"
report_path = os.path.join(OUTPUT_DIR, report_filename)
csv_output_path = os.path.join(OUTPUT_DIR, f"{REPORT_FILENAME_BASE}{report_suffix}_details.csv")

with open(report_path, "w") as report_file:
    def log_top(msg):
        print(msg)
        report_file.write(msg + "\n")

    log_top(
        f"\n--- GNN Clustering Report "
        f"({datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}) ---"
    )
    log_top(f"Database: {SQLITE_DB_PATH}")
    log_top(f"Clustering method: {CLUSTERING_METHOD}")
    if CLUSTERING_METHOD == 'static':
        log_top(f"  Static distance threshold: {chosen_distance_threshold}")
    else:
        log_top(f"  Dynamic clustering parameters: {CLUSTERING_PARAMS}")
    log_top(f"Hit weight: {HIT_GENE_WEIGHT_FACTOR}")
    log_top(f"Direct neighbor weight: {DIRECT_NEIGHBOR_WEIGHT_FACTOR}")
    log_top(
        "Mode: "
        + ("SSN-separated" if DIFFERENTIATE_BY_SSN_CLUSTER else "all neighborhoods together")
    )
    if COLLAPSE_IDENTICAL_NEIGHBORHOODS_ACTIVE:
        log_top("Collapsing enabled:")
        log_top(f"  Stage1 (core) thr: {COLLAPSE_CORE_SIMILARITY_THRESHOLD_ACTIVE}")
        log_top(f"  Stage2 (full) thr: {COLLAPSE_FULL_NEIGHBORHOOD_SIMILARITY_THRESHOLD_ACTIVE}")
    else:
        log_top("Collapsing disabled.")

    log_top(
        "Distance parallelism: "
        + ("joblib (enabled)" if PARALLELIZE_PDIST_ENABLED else "sequential")
    )
    if MAX_NEIGHBORS_PER_SIDE_ACTIVE is None:
        log_top("Neighbor window: disabled (all neighbors used).")
    else:
        log_top(f"Neighbor window: +/- {MAX_NEIGHBORS_PER_SIDE_ACTIVE} per side (closest kept).")
    log_top(
        f"Label policy: max={PLOT_MAX_LABELS}, min/cluster={PLOT_MIN_LABELS_PER_CLUSTER}, "
        f"extra/cluster={PLOT_EXTRA_LABELS_PER_CLUSTER}, sep={PLOT_MIN_LABEL_SEPARATION_DEG}deg, "
        f"min_angle={PLOT_MIN_LABEL_ANGLE_DEG}deg"
    )
    log_top(f"OMP_NUM_THREADS: {os.environ.get('OMP_NUM_THREADS', 'not set')}")
    if ORIGINAL_INPUT_SEQUENCE_ID:
        log_top(
            f"Highlight accession: {ORIGINAL_INPUT_SEQUENCE_ID} "
            f"(color {HIGHLIGHT_COLOR})"
        )
    else:
        log_top("No highlight accession set.")
    log_top(f"Plots -> {OUTPUT_DIR} as {OUTPUT_FORMATS}, dpi={DPI}")
    log_top(f"Report -> {report_path}")
    log_top(f"CSV details -> {csv_output_path}")
    log_top("-" * 70)

    clusters_by_ssn, final_labels_map, collapsed_groups_report = cluster_gene_neighborhoods_from_sqlite(
        db_path=SQLITE_DB_PATH,
        genes_table=GENES_TABLE,
        neighbors_table=NEIGHBORS_TABLE,
        differentiate_by_ssn_cluster=DIFFERENTIATE_BY_SSN_CLUSTER,
        ssn_cluster_value_to_filter=DEFAULT_SSN_CLUSTER_VALUE_TO_FILTER,
        collapse_identical_neighborhoods=COLLAPSE_IDENTICAL_NEIGHBORHOODS_ACTIVE,
        collapse_core_similarity_threshold=COLLAPSE_CORE_SIMILARITY_THRESHOLD_ACTIVE,
        collapse_full_neighborhood_similarity_threshold=COLLAPSE_FULL_NEIGHBORHOOD_SIMILARITY_THRESHOLD_ACTIVE,
        max_neighbors_per_side=MAX_NEIGHBORS_PER_SIDE_ACTIVE,
        original_input_sequence_id=ORIGINAL_INPUT_SEQUENCE_ID,
        distance_threshold=chosen_distance_threshold,
        clustering_method=CLUSTERING_METHOD,
        clustering_params=CLUSTERING_PARAMS,
        plot_max_labels=PLOT_MAX_LABELS,
        plot_min_labels_per_cluster=PLOT_MIN_LABELS_PER_CLUSTER,
        plot_extra_labels_per_cluster=PLOT_EXTRA_LABELS_PER_CLUSTER,
        plot_min_label_separation_deg=PLOT_MIN_LABEL_SEPARATION_DEG,
        plot_min_label_angle_deg=PLOT_MIN_LABEL_ANGLE_DEG,
        plot_max_stagger_levels=PLOT_MAX_STAGGER_LEVELS,
        plot_label_base_offset=PLOT_LABEL_BASE_OFFSET,
        plot_label_stagger_step=PLOT_LABEL_STAGGER_STEP,
        plot_draw_label_leaders=PLOT_DRAW_LABEL_LEADERS,
        plot_leader_min_offset=PLOT_LEADER_MIN_OFFSET,
        csv_output_path=csv_output_path,
        save_plots=SAVE_PLOTS,
        output_dir=OUTPUT_DIR,
        output_formats=OUTPUT_FORMATS,
        dpi=DPI,
        report_file_handle=report_file,
        parallelize_pdist=PARALLELIZE_PDIST_ENABLED,
    )

    if clusters_by_ssn:
        log_top("\n--- Final clustering results ---")
        for ssn_id, clusters_in_ssn in sorted(clusters_by_ssn.items(), key=lambda x: str(x[0])):
            log_top(f"\n### SSN {ssn_id} ###")
            if not clusters_in_ssn:
                log_top("  No clusters.")
                continue
            for cid, nh_list in sorted(clusters_in_ssn.items()):
                log_top(f"  Cluster {cid}: {len(nh_list)} neighborhoods")
                for nh in nh_list:
                    org, hit_id, _, acc, collapsed_info = final_labels_map.get(
                        nh, ("UNKNOWN", "UNKNOWN", None, "UNKNOWN", None)
                    )
                    highlight = " (ORIGINAL)" if ORIGINAL_INPUT_SEQUENCE_ID and acc == ORIGINAL_INPUT_SEQUENCE_ID else ""
                    collapsed_suffix = ""
                    if collapsed_info:
                        count, code = collapsed_info
                        collapsed_suffix = f" (Collapsed: {count}, code: {code})"
                    log_top(
                        f"    - {org} | Acc: {acc}{highlight}{collapsed_suffix} "
                        f"(hit_id={hit_id}, nh={nh})"
                    )
            log_top("  " + "-" * 30)

        if collapsed_groups_report:
            log_top("\n--- Collapsed neighborhood groups ---")
            for code, info in sorted(collapsed_groups_report.items()):
                log_top(
                    f"  Group {code}: rep={info['representative']} "
                    f"(n={info['count']})"
                )
                for nh in info["members"]:
                    org, hit_id, _, acc, _ = final_labels_map.get(
                        nh, ("UNKNOWN", "UNKNOWN", None, "UNKNOWN", None)
                    )
                    log_top(
                        f"    - {org} | Acc: {acc} (hit_id={hit_id}, nh={nh})"
                    )
            log_top("-" * 60)
    else:
        log_top("\nNo clusters formed at all.")

    log_top("\n--- Report end ---")


--- GNN Clustering Report (2026-03-23 09:55:46) ---
Database: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\CaMES\CaMES_10kBlast_10e_50eEdge_noFilter_300AST_min900AA_withoutEgtD_withoutMethyltrans_10N\39061_CaMES_10kBlast_10e_50eEdge_noFilter_300AST_min900AA_withoutEgtD_withoutMethyltrans_10N.sqlite
Clustering method: combined
  Dynamic clustering parameters: {}
Hit weight: 10
Direct neighbor weight: 3
Mode: all neighborhoods together
Collapsing enabled:
  Stage1 (core) thr: 0.0
  Stage2 (full) thr: 0.2
Distance parallelism: joblib (enabled)
Neighbor window: +/- 10 per side (closest kept).
Label policy: max=26, min/cluster=1, extra/cluster=1, sep=40.0deg, min_angle=6.5deg
OMP_NUM_THREADS: 12
Highlight accession: A0A7V4WV16 (color red)
Plots -> gnn_cluster_plots_circular as ['pdf'], dpi=600
Report -> gnn_cluster_plots_circular\gnn_clustering_report_all_neighborhoods.txt
CSV details -> gnn_cluster_plots_circular\gnn_clustering_report_all_neighborhoods_details.csv
-----------

Extracting features for 37 neighborhoods ...


  Neighborhoods: 100%|██████████| 37/37 [00:00<00:00, 999.75nh/s]


Feature extraction complete.
Applied side-aware neighbor cap (+/-10): removed 319 outer neighbors across 23 neighborhoods.

Raw SSN id distribution:
  SSN 1: 35 neighborhoods
  SSN 2: 1 neighborhoods
  SSN 3: 1 neighborhoods
------------------------------------------------------------
[Collapsing]  Collapsing neighborhoods (core thr: 0.0, full thr: 0.2)
[Collapsing]  Stage1: building core sparse matrix


[Collapsing]  Stage1: computing core Jaccard distances
  Pre-computing feature sets for N=37 ...


...

Processing SSN All_Neighborhoods with 37 neighborhoods.

--- SSN All_Neighborhoods: 37 neighborhoods ---


  SSN All_Neighborhoods: computing Jaccard distances ...
  Pre-computing feature sets for N=37 ...


  Done in 0.00s
  Using parallel Jaccard (N=37, cores=12)
    distance calc in 1.07s
  SSN All_Neighborhoods: linkage ...
    linkage in 0.00s
  SSN All_Neighborhoods: determining optimal clusters (method: combined) ...
    combined method -> 6 clusters
    individual method estimates: [11, 2, 6]
    clustering complete in 0.00s


meta NOT subset; don't know how to subset; dropped
meta NOT subset; don't know how to subset; dropped
Clustering SSN groups: 100%|██████████| 1/1 [00:02<00:00,  2.77s/group]

--- SSN All_Neighborhoods done in 2.70s ---
Detailed clustering table written to: gnn_cluster_plots_circular\gnn_clustering_report_all_neighborhoods_details.csv (37 rows)

Total runtime: 5.23s

--- Final clustering results ---

### SSN All_Neighborhoods ###
  Cluster 1: 31 neighborhoods
    - Acidobacteriota bacterium. | Acc: A0A399XDS8 (hit_id=QEUP01000041, nh=Acidobacteriota bacterium._QEUP01000041)
    - Actinomycetes bacterium. | Acc: A0A662DMY1 (hit_id=QMQC01000053, nh=Actinomycetes bacterium._QMQC01000053)
    - Alteromonadaceae bacterium. | Acc: A0A2A5HFC2 (hit_id=NVXL01000214, nh=Alteromonadaceae bacterium._NVXL01000214)
    - Bdellovibrionales bacterium GWB1_55_8. | Acc: A0A1F3T9T9 (hit_id=MEPR01000010, nh=Bdellovibrionales bacterium GWB1_55_8._MEPR01000010)
    - Caldithrix abyssi. | Acc: A0A7V4WV16 (ORIGINAL) (hit_id=DRQG01000086, nh=Caldithrix abyssi._DRQG01000086)
    - Calditrichota bacterium. | Acc: A0A849M3X8 (hit_id=JABFGF010000001, nh=Calditrichota bacterium._JABFGF01

In [3]:
import sqlite3
import pandas as pd

# Path to your SQLite database
db_path = r"D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\CaMES\CaMES_10kBlast_10e_50eEdge_noFilter_300AST_min900AA_withoutEgtD_withoutMethyltrans_10N\39061_CaMES_10kBlast_10e_50eEdge_noFilter_300AST_min900AA_withoutEgtD_withoutMethyltrans_10N.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Query to count occurrences of each taxon_id and get id and organism from attributes
query = """
    SELECT 
        a.id,
        a.organism,
        n.taxon_id,
        COUNT(*) as neighbor_count
    FROM neighbors n
    LEFT JOIN attributes a ON n.taxon_id = a.taxon_id
    GROUP BY n.taxon_id, a.id, a.organism
    ORDER BY neighbor_count DESC
"""

# Execute query and load into DataFrame
results_df = pd.read_sql_query(query, conn)

# Close connection
conn.close()

# Display results
print("Number of neighbors per taxon_id with enzyme info:")
print(results_df)

# Optionally save to CSV
results_df.to_csv("neighbor_counts_with_info.csv", index=False)
print("\nResults saved to neighbor_counts_with_info.csv")

Number of neighbors per taxon_id with enzyme info:
                 id                                  organism taxon_id   
0   JABFGF010000001                  Calditrichota bacterium.  2212469  \
1      QQUE01000012                  Calditrichota bacterium.  2212469   
2      RFFS01000159                  Calditrichota bacterium.  2212469   
3      RFIL01000057                  Calditrichota bacterium.  2212469   
4      LADR01000008               Desulfatitalea sp. BRH_c12.  1629708   
5      WVYX01000149               Gemmatimonadales bacterium.  2448054   
6   JACPUR010000013                Tectimicrobiota bacterium.  2528274   
7      BJTG01000010          Anaeromyxobacter diazotrophicus.  2590199   
8      FMWD01000006              Thiohalomonas denitrificans.   415747   
9          LO017727      Magnetococcus massalia (strain MO-1)   451514   
10         CP003051             Thioflavicoccus mobilis 8321.   765912   
11     DSCF01000035                    Thermoprotei archaeon.